# ️ Glu-Stock: 01_INFERENCE_ENGINE
**Phase**: Quantitative Scanning & ML Intelligence | v18.26 (CNN-LLM Dual Core)

This notebook uses **CNN Visual Intelligence** as the primary technical gate. 
**v18.26**: Decommissioned LGBM to increase signal recall. Threshold set to 0.45 for high-flow audit.

In [ ]:
#!pip install -q yfinance pandas tensorflow ta firebase-admin


In [ ]:
import os, json, numpy as np, pandas as pd, yfinance as yf, warnings, ta, firebase_admin
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from firebase_admin import credentials, firestore
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()
    def predict(self, df):
        try:
            data = df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(30).values
            norm = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-7)
            self.interpreter.set_tensor(self.interpreter.get_input_details()[0]['index'], np.expand_dims(norm.astype(np.float32), axis=0))
            self.interpreter.invoke()
            out = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(out[1]) if len(out) > 1 else float(out[0])
        except: return 0.5


In [ ]:
def run_inference_engine():
    print("Starting 01_INFERENCE_ENGINE (v18.26 [DUAL-CORE])...\n")
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    cred_json = json.loads(secrets.get_secret("FIREBASE_KEY_JSON"))
    if not firebase_admin._apps: firebase_admin.initialize_app(credentials.Certificate(cred_json))
    db = firestore.client()
    
    cnn = CNNPredictor('/kaggle/input/notebooks/permanalwep/glu-stock-cnn-00b/cnn_daily_t2.tflite')
    
    # 2. Universal Scanner (Full IDX List)
    cohort = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']
    data = yf.download(cohort + ['^JKSE'], period='2y', progress=False, auto_adjust=True)
    
    candidates = []
    for ticker in cohort:
        try:
            hist = data.loc[:, (slice(None), ticker)].dropna()
            hist.columns = hist.columns.droplevel(1)
            if len(hist) < 150: continue
            if hist['Close'].iloc[-1] > hist['Close'].tail(50).mean():
                score = cnn.predict(hist)
                if score >= 0.45:
                    candidates.append({ticker: {'price': float(hist['Close'].iloc[-1]), 'score': score}})
                    print(f"[CANDIDATE] {ticker} @ {score:.2%}")
        except: continue
    
    if candidates:
        for cand in candidates: db.collection("glu_stock_queue_signals").add({"payload": cand, "timestamp": datetime.now().isoformat()})
        print(f"\n[SUCCESS] {len(candidates)} candidates sent to Strategic Brain (LLM).")
    else:
        print("\n[IDLE] No technical patterns detected.")

run_inference_engine()
